# Classificador de imagens CIFAR-10

Adaptado de `Classificador_Imagens (2).ipynb`, fornecido pelo professor:
CNN com BatchNormalization, Dropout, EarlyStopping, curvas de acurácia e loss,
avaliação no teste e predições. Os resultados devem ser gerados por execução
local. A primeira execução baixa o CIFAR-10; o treino completo pode demorar.


In [ ]:
# Biblioteca para operações numéricas
import numpy as np

# Para visualização
import matplotlib.pyplot as plt

# Para redes neurais
import tensorflow as tf

# Interface de alto nível para construir redes neurais
from tensorflow import keras

# Importando as camadas da rede
from tensorflow.keras.layers import Conv2D, Input, MaxPooling2D, \
 Flatten, Dense, Dropout

# Funções utilitárias para trabalhar com imagens
from tensorflow.keras.utils import load_img, img_to_array

# Carregar a base de Dados CIFAR-10

In [ ]:
# Carregar a base CIFAR
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Normalizar os valores para um intervalo [0,1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# y vem como coluna.
y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = ["avião", "automóvel", "pássaro", "gato", "cervo",
               "cachorro", "sapo", "cavalo", "navio", "caminhão"]

print(f"Formato de X_train: ", X_train.shape)
print(f"Formato de y_train: ", y_train.shape)
print(f"Formato de X_test: ", X_test.shape)
print(f"Formato de y_test: ", y_test.shape)



# Plotar Algumas Imagens

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.suptitle("Exemplos da base CIFAR-10.")
plt.tight_layout()
plt.show()

# Criação do Modelo CNN

In [ ]:
from tensorflow import keras
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Dropout,
    Flatten,
    Dense
)

model = keras.Sequential()

# Entrada
model.add(Input(shape=(32, 32, 3)))

# -------------------------
# Bloco 1
# -------------------------
model.add(Conv2D(
    filters=32,
    kernel_size=(3, 3),
    padding='same',
    activation='relu'
))
model.add(BatchNormalization())

model.add(Conv2D(
    filters=32,
    kernel_size=(3, 3),
    padding='same',
    activation='relu'
))
model.add(BatchNormalization())

model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))


# -------------------------
# Bloco 2
# -------------------------
model.add(Conv2D(
    filters=64,
    kernel_size=(3, 3),
    padding='same',
    activation='relu'
))
model.add(BatchNormalization())

model.add(Conv2D(
    filters=64,
    kernel_size=(3, 3),
    padding='same',
    activation='relu'
))
model.add(BatchNormalization())

model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))


# -------------------------
# Bloco 3
# -------------------------
model.add(Conv2D(
    filters=128,
    kernel_size=(3, 3),
    padding='same',
    activation='relu'
))
model.add(BatchNormalization())

model.add(Conv2D(
    filters=128,
    kernel_size=(3, 3),
    padding='same',
    activation='relu'
))
model.add(BatchNormalization())

model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.30))


# -------------------------
# CNN → Rede totalmente conectada
# -------------------------
model.add(Flatten())

model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

# CIFAR-10 possui 10 classes
model.add(Dense(10, activation='softmax'))


# -------------------------
# Compilação
# -------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Treinar a Rede

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3, # Se a val_loss não melhorar por 3 épocas consecutivas, o treinamento para.
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    validation_split=0.2,
    callbacks=[early_stopping]
)

# Plot da Acurácia e Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label="Treino")
plt.plot(history.history['val_accuracy'], label="Validação")
plt.title("Acurácia por época")
plt.xlabel("Épocas")
plt.ylabel("Acurácia")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label="Treino")
plt.plot(history.history['val_loss'], label="Validação")
plt.title("Loss por época")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Realizando Predições

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc:.2}")
print(f"Loss no test: {test_loss:.4f}")

# Previsões em Imagens do Teste

In [ ]:
pred_probs = model.predict(X_test[:9])
pred_labels = np.argmax(pred_probs, axis=1)
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(X_test[i])
    real = class_names[y_test[i]]
    pred = class_names[pred_labels[i]]
    plt.title(f"Real: {real}\nPredição: {pred}")
    plt.axis('off')
plt.suptitle("Previsões em imagens do conjunto de teste")
plt.tight_layout()
plt.show()